In [1]:
import re
import random
from collections import Counter
from typing import List

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

import urllib.parse
import html


In [2]:
# Config
DATA_PATH = 'data/XSS_dataset.csv'
MAX_VOCAB = 15000
MAX_LEN = 120
BATCH_SIZE = 64
EPOCHS = 30
LR = 1e-3
PATIENCE = 5
SEED = 42
MODEL_SAVE_PATH = 'xss_model.pth'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [3]:
# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [4]:
# def xss_preprocess(text: str) -> str:
#     if pd.isna(text):
#         return ''
#     s = str(text).strip()
#     s = s.replace('\r', ' ').replace('\n', ' ')
#     s = s.replace('\x00', '').replace('\0', '')
#     s = urllib.parse.unquote(s)
#     s = html.unescape(s)
#     s = re.sub(r'\\u([0-9a-fA-F]{4})',
#                lambda m: chr(int(m.group(1), 16)), s)
#     s = re.sub(r'\s+', ' ', s).strip()
#     s = s.lower()
#     s = s.replace('<', ' < ').replace('>', ' > ')
#     s = re.sub(r'\s+', ' ', s).strip()
#     return s
def xss_preprocess(text: str) -> str:
    if pd.isna(text):
        return ''
    s = str(text).strip()
    s = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]+', ' ', s)
    s = urllib.parse.unquote(s)
    s = html.unescape(s)
    s = re.sub(r'\\u([0-9a-fA-F]{4})',
               lambda m: chr(int(m.group(1), 16)), s)
    s = s.lower()
    s = s.replace('<', ' < ').replace('>', ' > ')
    s = re.sub(r'\s+', ' ', s).strip()
    return s



In [5]:
def xss_tokenize(text: str) -> List[str]:
    pattern = re.compile(r"\w+|</?\w+|[<>\"'=;:/.\-]|\S")
    return pattern.findall(text)[:MAX_LEN]


In [6]:
df = pd.read_csv(DATA_PATH)

texts = df['sentence'].astype(str).values
labels = df['Label'].astype(int).values
# 60% train | 20% val | 20% test
train_texts, temp_texts, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.4, stratify=labels, random_state=SEED
)

val_texts, test_texts, y_val, y_test = train_test_split(
    temp_texts, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED
)

print(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")


Train: 8211 | Val: 2737 | Test: 2738


In [7]:
train_tokens = [xss_tokenize(xss_preprocess(t)) for t in train_texts]
val_tokens   = [xss_tokenize(xss_preprocess(t)) for t in val_texts]


In [8]:
counter = Counter()
for tokens in train_tokens:
    counter.update(tokens)

vocab = ['<PAD>', '<UNK>'] + [w for w, _ in counter.most_common(MAX_VOCAB)]
word2idx = {w: i for i, w in enumerate(vocab)}

print("Vocab size:", len(vocab))


Vocab size: 6614


In [9]:
def encode_split(tokens_list: List[List[str]]) -> List[List[int]]:
    sequences = []
    for tokens in tokens_list:
        seq = [word2idx.get(t, word2idx['<UNK>']) for t in tokens]
        seq = seq[:MAX_LEN]
        seq += [word2idx['<PAD>']] * (MAX_LEN - len(seq))
        sequences.append(seq)
    return sequences
train_seqs = encode_split(train_tokens)
val_seqs   = encode_split(val_tokens)


In [10]:
class XSSDataset(Dataset):
    def __init__(self, seqs, labels):
        self.seqs = seqs
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.LongTensor(self.seqs[idx]),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )


In [11]:
counts = np.bincount(y_train)
weights = 1.0 / torch.tensor(counts, dtype=torch.float)
sample_weights = weights[y_train]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
train_loader = DataLoader(
    XSSDataset(train_seqs, y_train),
    batch_size=BATCH_SIZE,
    sampler=sampler
)

val_loader = DataLoader(
    XSSDataset(val_seqs, y_val),
    batch_size=BATCH_SIZE,
    shuffle=False
)


In [12]:
class XSSDetector(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 128, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(128, 128, k, padding=k//2) for k in [3,5,7]
        ])
        self.dropout = nn.Dropout(0.4)
        self.lstm = nn.LSTM(128*3, 256,
                            batch_first=True,
                            bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.emb(x).permute(0,2,1)
        x = torch.cat([F.relu(c(x)) for c in self.convs], dim=1)
        x = x.permute(0,2,1)
        x = self.dropout(x)
        x, _ = self.lstm(x)
        x = torch.max(x, dim=1)[0]
        return self.fc(x)


In [13]:
model = XSSDetector(len(vocab)).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights.to(DEVICE))
optimizer = optim.AdamW(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2
)


In [14]:
best_auc = -1
patience_cnt = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_probs = torch.softmax(
            model(torch.stack([x for x,_ in val_loader.dataset]).to(DEVICE)),
            dim=1
        )[:,1].cpu().numpy()

    val_auc = roc_auc_score(y_val, val_probs)
    scheduler.step(val_auc)

    print(f"Epoch {epoch:02d} | Val AUC: {val_auc:.5f}")

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(
            {'model': model.state_dict(), 'word2idx': word2idx},
            MODEL_SAVE_PATH
        )
        patience_cnt = 0
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print("Early stopping!")
            break


Epoch 01 | Val AUC: 0.99991
Epoch 02 | Val AUC: 0.99994
Epoch 03 | Val AUC: 0.99996
Epoch 04 | Val AUC: 0.99998
Epoch 05 | Val AUC: 0.99998
Epoch 06 | Val AUC: 1.00000
Epoch 07 | Val AUC: 1.00000
Epoch 08 | Val AUC: 0.99999
Epoch 09 | Val AUC: 1.00000
Epoch 10 | Val AUC: 1.00000
Epoch 11 | Val AUC: 1.00000
Epoch 12 | Val AUC: 1.00000
Epoch 13 | Val AUC: 1.00000
Epoch 14 | Val AUC: 1.00000
Early stopping!


In [15]:
ckpt = torch.load(MODEL_SAVE_PATH)
model.load_state_dict(ckpt['model'])
word2idx = ckpt['word2idx']
model.eval()
test_tokens = [xss_tokenize(xss_preprocess(t)) for t in test_texts]
test_seqs   = encode_split(test_tokens)
test_loader = DataLoader(
    XSSDataset(test_seqs, y_test),
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [16]:
with torch.no_grad():
    test_probs = torch.softmax(
        model(torch.stack([x for x,_ in test_loader.dataset]).to(DEVICE)),
        dim=1
    )[:,1].cpu().numpy()

auc = roc_auc_score(y_test, test_probs)
acc = accuracy_score(y_test, (test_probs > 0.5).astype(int))

print(f"TEST → AUC: {auc:.5f} | ACC: {acc:.5f}")
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, (test_probs > 0.5).astype(int)))


TEST → AUC: 1.00000 | ACC: 0.99854
[[1260    3]
 [   1 1474]]


In [17]:
def test_on_xss_payload_list_txt(txt_path: str = 'xss-payload-list.txt',
                                  batch_size: int = 64):
    """
    Test the model on a list of known XSS payloads (txt file). All payloads are assumed to be malicious.
    """
    print(f"\n=== test model on XSS payload list ({txt_path}) ===")


    with open(txt_path, 'r', encoding='utf-8', errors='ignore') as f:
        payloads = [line.strip().strip('"').strip("'") for line in f
                    if line.strip() and not line.startswith('#')]

    total_samples = len(payloads)
    print(f"Number of payloads loaded: {total_samples:,}")


    labels = np.array([1] * total_samples)

    print("sample payloads::")
    for i in range(min(5, total_samples)):
        print(f"   {i+1}: {payloads[i][:100]}...")


    print("\n Preprocessing and tokenization...")
    clean_texts = [xss_preprocess(p) for p in payloads]
    tokens_list = [xss_tokenize(t) for t in clean_texts]
    sequences = encode_split(tokens_list)


    print(f"\n Forecasting with batches of {batch_size}... ")
    model.eval()
    probs_list = []

    with torch.no_grad():
        for i in range(0, len(sequences), batch_size):
            batch = torch.LongTensor(sequences[i:i + batch_size]).to(DEVICE)
            batch_probs = torch.softmax(model(batch), dim=1)[:, 1].cpu().numpy()
            probs_list.append(batch_probs)

            if (i // batch_size + 1) % 50 == 0 or i + batch_size >= len(sequences):
                processed = min(i + batch_size, len(sequences))
                print(f"   Processed: {processed:,}/{total_samples:,}")

    probs = np.concatenate(probs_list)
    preds = (probs > 0.5).astype(int)


    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

    acc = accuracy_score(labels, preds)
    recall = recall_score(labels, preds)
    precision = precision_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    #auc = roc_auc_score(labels, probs)

    print("\n" + "="*60)
    print("Results on the XSS payload list (all Malicious):")
    print("="*60)
    print(f"number of payloads:{total_samples:,}")
    print(f"Accuracy:  {acc:.5f}  ")
    print(f"Recall:    {recall:.5f}  ")
    print(f"Precision: {precision:.5f}")
    print(f"F1-Score:  {f1:.5f}")
    #print(f"AUC:       {auc:.5f}")
    print("="*60)


    missed = np.sum((preds == 0) & (labels == 1))
    print(f"Number of payloads that the model did not recognize (Missed){missed:,} at {total_samples:,}")

    if missed > 0:
        print("\nSome payloads that the model misidentified (Benign said):")
        missed_indices = np.where((preds == 0) & (labels == 1))[0]
        for idx in missed_indices[:10]:  # حداکثر ۱۰ تا
            print(f"   Probability: {probs[idx]:.4f} | Payload: {payloads[idx][:100]}...")

    return {
        'num_payloads': total_samples,
        'missed': missed,
        'recall': recall,
        #'auc': auc,
        'probs': probs,
        'preds': preds
    }

In [18]:
results = test_on_xss_payload_list_txt('data/xss-payload-list.txt')


=== test model on XSS payload list (data/xss-payload-list.txt) ===
Number of payloads loaded: 6,613
sample payloads::
   1: -prompt(8)-...
   2: -prompt(8)-...
   3: ;a=prompt,a()//...
   4: ;a=prompt,a()//...
   5: -eval("window['pro'%2B'mpt'](8)")-...

 Preprocessing and tokenization...

 Forecasting with batches of 64... 
   Processed: 3,200/6,613
   Processed: 6,400/6,613
   Processed: 6,613/6,613

Results on the XSS payload list (all Malicious):
number of payloads:6,613
Accuracy:  0.98231  
Recall:    0.98231  
Precision: 1.00000
F1-Score:  0.99107
Number of payloads that the model did not recognize (Missed)117 at 6,613

Some payloads that the model misidentified (Benign said):
   Probability: 0.0103 | Payload: -prompt(8)-...
   Probability: 0.0103 | Payload: -prompt(8)-...
   Probability: 0.0001 | Payload: -eval("window['pro'%2B'mpt'](8)")-...
   Probability: 0.0001 | Payload: -eval("window['pro'%2B'mpt'](8)")-...
   Probability: 0.2100 | Payload: <base href=//0>...
   Probabili

In [26]:
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

def test_on_new_dataset(
    csv_path: str,
    model,
    word2idx: dict,
    text_col: str = "sentence",
    label_col: str = "Label",
    has_label: bool = True,
    threshold: float = 0.8,
    batch_size: int = 128
):
    df = pd.read_csv(csv_path)

    texts = df[text_col].astype(str).values
    labels = df[label_col].values if has_label else None

    # preprocess + tokenize
    tokens = [xss_tokenize(xss_preprocess(t)) for t in texts]
    seqs = encode_split(tokens)

    model.eval()
    probs = []

    with torch.no_grad():
        for i in range(0, len(seqs), batch_size):
            batch = torch.LongTensor(seqs[i:i+batch_size]).to(DEVICE)
            p = torch.softmax(model(batch), dim=1)[:, 1]
            probs.extend(p.cpu().numpy())

    probs = np.array(probs)
    preds = (probs >= threshold).astype(int)

    print("📊 Prediction summary")
    print(pd.Series(preds).value_counts())

    if has_label:
        print("\n✅ Evaluation metrics")
        print("AUC:", roc_auc_score(labels, probs))
        print("Accuracy:", accuracy_score(labels, preds))
        print("\nConfusion Matrix:")
        print(confusion_matrix(labels, preds))
        print("\nClassification Report:")
        print(classification_report(labels, preds, digits=4))

    # append results
    df["xss_prob"] = probs
    df["xss_pred"] = preds

    return df


In [27]:
# model.load_state_dict(torch.load("best_update_xss_model.pth"))
# model.to(DEVICE)
ckpt = torch.load("xss_model.pth", map_location=DEVICE)

model.load_state_dict(ckpt["model"])
word2idx = ckpt["word2idx"]

model.to(DEVICE)
model.eval()


df_results = test_on_new_dataset(
    csv_path="data/xss_da.csv",
    model=model,
    word2idx=word2idx,
    has_label=True
)

df_results.to_csv("xss_test_results_cnn_bilstm.csv", index=False)


📊 Prediction summary
0    22874
1    18872
Name: count, dtype: int64

✅ Evaluation metrics
AUC: 0.9982935352289133
Accuracy: 0.7243568246059503

Confusion Matrix:
[[22870 11503]
 [    4  7369]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9998    0.6653    0.7990     34373
           1     0.3905    0.9995    0.5616      7373

    accuracy                         0.7244     41746
   macro avg     0.6951    0.8324    0.6803     41746
weighted avg     0.8922    0.7244    0.7571     41746



In [22]:
from sklearn.metrics import f1_score

best_t, best_f1 = 0, 0
for t in np.linspace(0.1, 0.99, 50):
    preds = (val_probs > t).astype(int)
    f1 = f1_score(y_val, preds)
    if f1 > best_f1:
        best_f1, best_t = f1, t

print('tersh:', best_t), best_f1


tersh: 0.1


(None, 0.9989840839823908)